In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# =========================
# DEVICE
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# =========================
# LOAD FEATURES
# =========================
vision_features = torch.load('../Encoder/features.pt').to(device)
labels = torch.load('../Encoder/labels.pt')

In [5]:
# =========================
# TOKENIZERS
# =========================
text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
llm_tokenizer = AutoTokenizer.from_pretrained("gpt2")

llm_tokenizer.pad_token = llm_tokenizer.eos_token

c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\minhp\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [6]:
# =========================
# MODELS (LIGHTWEIGHT)
# =========================
print("Loading text encoder...")
text_encoder = AutoModel.from_pretrained("bert-base-uncased").to(device)

print("Loading LLM...")
llm = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

print("Models loaded.")

# Freeze models
for p in text_encoder.parameters():
    p.requires_grad = False

for p in llm.parameters():
    p.requires_grad = False


Loading text encoder...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5658.03it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading LLM...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6913.82it/s]


Models loaded.


In [7]:

# =========================
# DIMENSIONS
# =========================
vision_dim = vision_features.shape[1]
text_dim = text_encoder.config.hidden_size
hidden_dim = 512
llm_dim = llm.config.n_embd

In [8]:

# =========================
# PROJECTION LAYERS
# =========================
image_projection = nn.Linear(vision_dim, hidden_dim).to(device)
text_projection = nn.Linear(text_dim, hidden_dim).to(device)
fusion_projection = nn.Linear(hidden_dim * 2, hidden_dim).to(device)
llm_projection = nn.Linear(hidden_dim, llm_dim).to(device)

In [9]:

# =========================
# CONTEXT TOKEN
# =========================
clinical_context = nn.Parameter(torch.randn(1, hidden_dim)).to(device)

In [10]:

# =========================
# TEXT ENCODING
# =========================
def encode_text(text_inputs):
    inputs = text_tokenizer(
        text_inputs,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = text_encoder(**inputs)

    return outputs.last_hidden_state[:, 0, :]  # CLS token

In [11]:
# =========================
# FUSION
# =========================
def fuse(vision_emb, text_emb):
    fused = torch.cat([vision_emb, text_emb], dim=-1)
    return fusion_projection(fused)

In [12]:
# =========================
# PROMPT
# =========================
def build_prompt():
    return "Clinical Impression: Liver CT scan shows "

In [13]:
# =========================
# GENERATION (IMPROVED)
# =========================
def generate_text(vision_feats, max_len=80):

    # ---- Vision ----
    vision_emb = image_projection(vision_feats)

    # ---- Context ----
    context = clinical_context.expand(vision_emb.shape[0], -1)

    # ---- Fuse ----
    fused = fuse(vision_emb, context)

    # ---- Project to LLM space ----
    vision_token = llm_projection(fused).unsqueeze(1)

    # ---- Prompt ----
    prompt = build_prompt()
    input_ids = llm_tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    prompt_embeds = llm.transformer.wte(input_ids)

    # Combine vision + text
    generated = torch.cat([vision_token, prompt_embeds], dim=1)

    output_tokens = []

    for step in range(max_len):
        with torch.no_grad():
            outputs = llm(inputs_embeds=generated)
            logits = outputs.logits[:, -1, :]

            # 🔥 sampling instead of greedy (better text)
            probs = torch.softmax(logits / 0.8, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            token_id = next_token.item()
            output_tokens.append(token_id)

            if token_id == llm_tokenizer.eos_token_id:
                break

            next_emb = llm.transformer.wte(next_token)
            generated = torch.cat([generated, next_emb], dim=1)

    if output_tokens:
        report = llm_tokenizer.decode(output_tokens, skip_special_tokens=True)
    else:
        report = "[No report generated]"

    return report

In [14]:
# =========================
# RUN
# =========================
if __name__ == "__main__":

    print("\n" + "="*70)
    print("LIGHTWEIGHT VLM MEDICAL REPORT GENERATOR")
    print("="*70)

    num_reports = min(3, len(vision_features))

    for idx in range(num_reports):
        print(f"\n[Report {idx+1}] Processing...")

        try:
            report = generate_text(
                vision_features[idx:idx+1],
                max_len=100
            )

            print("\n" + "─"*70)
            print(f"CLINICAL IMPRESSION - Image {idx+1}")
            print("─"*70)
            print(report)
            print("─"*70)

        except Exception as e:
            print(f"Error: {e}")

    print("\nDone.")


LIGHTWEIGHT VLM MEDICAL REPORT GENERATOR

[Report 1] Processing...

──────────────────────────────────────────────────────────────────────
CLINICAL IMPRESSION - Image 1
──────────────────────────────────────────────────────────────────────
erythema, liver disease, and myalgia

Clinical Impression: Liver CT scan shows erythema, liver disease, and myalgia Disease: Blood test shows erythema, liver disease, and anorexia

Disorders: Severe dysarthria and hypothyroidism

Severe dysarthria and hypothyroidism Trauma: Sudden death, sudden amputation, or seizure

Sudden death, sudden amputation, or
──────────────────────────────────────────────────────────────────────

[Report 2] Processing...

──────────────────────────────────────────────────────────────────────
CLINICAL IMPRESSION - Image 2
──────────────────────────────────────────────────────────────────────
erythrocyte erythroid in men.

HIPAA ALA Alds Aldric Aldr Aldr Basic Culture 0.10% 0.10% 0.10% 0.10% 0.10% 0.10% 0.10% 0.10% 0.10% 0.